<a href="https://colab.research.google.com/github/mgcprojects/fraud-detection-deep-clustering/blob/main/fraud-detection-deep-clustering" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**CS7357 – Neural Networks and Deep Learning**

**Project Title:** Credit Card Fraud Detection using Neural Networks

**Team Members:** Dhruv Shrivastava, Mauricio Gonzalez, and Sukumar Muthusamy

**Dataset:** Kaggle Credit Card Fraud Detection

**Platform:** Google Colab (GPU Runtime)

In [ ]:
# ----------------------------------------------
# STEP 1: Mount Google Drive and verify dataset path
# ----------------------------------------------
# In this step, we:
# 1. Mount the user's Google Drive into the Colab environment.
# 2. Define the path where the dataset (creditcard.csv) is stored in Drive.
# 3. Verify that the dataset exists at the specified path before proceeding.
# This ensures that the dataset is accessible for all subsequent steps
# such as loading, preprocessing, and training.

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set the dataset path
DATA_PATH = "/content/drive/MyDrive/data/fraud_detection/creditcard.csv"

# Verify that the file exists
import os
if os.path.exists(DATA_PATH):
    print("✅ Dataset found at:", DATA_PATH)
else:
    print("⚠️ Dataset not found. Please check the path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset found at: /content/drive/MyDrive/data/fraud_detection/creditcard.csv


In [ ]:
# ----------------------------------------------
# STEP 2: Load the dataset and perform basic checks
# ----------------------------------------------
# In this step, we:
# 1. Load the credit card fraud dataset from Google Drive.
# 2. Inspect the shape, columns, and a few sample rows to confirm data integrity.
# 3. Check class distribution to understand the fraud vs. non-fraud imbalance.
# 4. Review 'Time' column statistics and verify if the dataset is already sorted chronologically.
# These checks prepare us for the next step: chronological splitting and preprocessing.

import pandas as pd
import numpy as np

# Load
df = pd.read_csv(DATA_PATH)

# Basic shape and columns
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))

# Peek at the data
display(df.head())

# Class distribution
class_counts = df['Class'].value_counts().sort_index()
fraud_rate = class_counts.get(1, 0) / class_counts.sum()
print("\nClass counts:\n", class_counts)
print(f"Fraud rate: {fraud_rate:.6f}")

# Time column sanity checks
print("\nTime column stats:")
print(df['Time'].describe())

# Check if already sorted by time and report range
is_sorted = df['Time'].is_monotonic_increasing
print("Is dataset already sorted by Time?", is_sorted)
print("Time range:", float(df['Time'].min()), "to", float(df['Time'].max()))

Shape: (284807, 31)

Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0



Class counts:
 Class
0    284315
1       492
Name: count, dtype: int64
Fraud rate: 0.001727

Time column stats:
count    284807.000000
mean      94813.859575
std       47488.145955
min           0.000000
25%       54201.500000
50%       84692.000000
75%      139320.500000
max      172792.000000
Name: Time, dtype: float64
Is dataset already sorted by Time? True
Time range: 0.0 to 172792.0


In [ ]:
# ----------------------------------------------
# STEP 3: Chronological split and scaling
# ----------------------------------------------
# In this step, we:
# 1) Verify the data is sorted by Time (it is, from the previous cell).
# 2) Split chronologically into 70% train, 15% validation, 15% test.
# 3) Standardize only the non-PCA features 'Time' and 'Amount' using stats from the train set.
# 4) Prepare X_train, X_val, X_test and y_* for the next SMOTE + model steps.

from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# 1) Ensure chronological order
assert df['Time'].is_monotonic_increasing, "Dataset must be sorted by Time for a temporal split."

n = len(df)
train_end = int(0.70 * n)
val_end   = int(0.85 * n)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

# 2) Separate features and labels
FEATURES = [c for c in df.columns if c != 'Class']
LABEL    = 'Class'

# 3) Standardize 'Time' and 'Amount' based on the training set only
scaler = StandardScaler()
for col in ['Time', 'Amount']:
    # Fit on train, transform train/val/test
    train_df[col] = scaler.fit_transform(train_df[[col]])
    val_df[col]   = scaler.transform(val_df[[col]])
    test_df[col]  = scaler.transform(test_df[[col]])

# 4) Build matrices for modeling
X_train = train_df[FEATURES].values
y_train = train_df[LABEL].values

X_val = val_df[FEATURES].values
y_val = val_df[LABEL].values

X_test = test_df[FEATURES].values
y_test = test_df[LABEL].values

# Quick summary to confirm sizes and class counts per split
def split_summary(name, y):
    uniq, cnt = np.unique(y, return_counts=True)
    counts = dict(zip(uniq, cnt))
    fraud = counts.get(1, 0)
    total = y.shape[0]
    rate  = fraud / total if total else 0.0
    print(f"{name}: n={total}, fraud={fraud}, rate={rate:.6f}")

print("Split sizes and fraud rates:")
split_summary("Train", y_train)
split_summary("Val  ", y_val)
split_summary("Test ", y_test)

Split sizes and fraud rates:
Train: n=199364, fraud=384, rate=0.001926
Val  : n=42721, fraud=56, rate=0.001311
Test : n=42722, fraud=52, rate=0.001217


In [ ]:
# ----------------------------------------------
# STEP 4: Apply SMOTE on the TRAINING SET ONLY
# ----------------------------------------------
# In this step, we:
# 1) Oversample the minority class (fraud = 1) on X_train, y_train using SMOTE.
# 2) Keep validation and test sets unchanged for fair evaluation.
# 3) Print the new class distribution after SMOTE.
#
# Note: Run the install cell for imbalanced-learn once if import fails.

# Try import, install if needed
try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    !pip -q install imbalanced-learn==0.12.3
    from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

# Check new balance
import numpy as np
def count_stats(y, name):
    uniq, cnt = np.unique(y, return_counts=True)
    counts = dict(zip(uniq, cnt))
    total = y.shape[0]
    fraud = counts.get(1, 0)
    rate = fraud / total if total else 0.0
    print(f"{name}: n={total}, fraud={fraud}, rate={rate:.4f}")

print("Class distribution after SMOTE:")
count_stats(y_train_sm, "Train (SMOTE)")
count_stats(y_val,      "Val (unchanged)")
count_stats(y_test,     "Test (unchanged)")

Class distribution after SMOTE:
Train (SMOTE): n=397960, fraud=198980, rate=0.5000
Val (unchanged): n=42721, fraud=56, rate=0.0013
Test (unchanged): n=42722, fraud=52, rate=0.0012


In [ ]:
# ----------------------------------------------
# STEP 5: Build and train the Baseline Neural Network
# ----------------------------------------------
# In this step, we:
# 1) Define a simple MLP with hidden sizes 64 -> 32 -> 16 and ReLU activations.
# 2) Use a single sigmoid output for binary classification.
# 3) Compile with binary cross-entropy loss and Adam optimizer.
# 4) Track validation AUC and stop early when it stops improving.
# 5) Train on the SMOTE-balanced TRAIN set and validate on the original VAL set.

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

model = keras.Sequential([
    layers.Input(shape=(X_train_sm.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name="auc")]
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=5, restore_best_weights=True
)

history = model.fit(
    X_train_sm, y_train_sm,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=2048,
    callbacks=[early_stop],
    verbose=1
)

print("Training complete. Best val AUC seen during training:",
      f"{max(history.history['val_auc']):.4f}")

Epoch 1/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - auc: 0.9820 - loss: 0.1789 - val_auc: 0.9681 - val_loss: 0.0138
Epoch 2/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.9995 - loss: 0.0275 - val_auc: 0.9448 - val_loss: 0.0094
Epoch 3/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - auc: 0.9997 - loss: 0.0107 - val_auc: 0.9278 - val_loss: 0.0078
Epoch 4/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.9998 - loss: 0.0061 - val_auc: 0.9191 - val_loss: 0.0071
Epoch 5/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - auc: 0.9998 - loss: 0.0042 - val_auc: 0.9193 - val_loss: 0.0066
Epoch 6/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - auc: 0.9999 - loss: 0.0032 - val_auc: 0.9104 - val_loss: 0.0062
Training complete. Best val AUC seen during training: 0.9681


In [ ]:
# ----------------------------------------------
# STEP 6: Evaluate on validation and test sets
# ----------------------------------------------
# In this step, we:
# 1) Get predicted probabilities from the model.
# 2) Use a default threshold of 0.5 to make class predictions.
# 3) Compute Precision, Recall, F1, and AUC-ROC for both VAL and TEST.
# 4) Show the confusion matrices and a short report.

import numpy as np
from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

def evaluate_split(name, X, y, thr=0.5):
    probs = model.predict(X, verbose=0).ravel()
    preds = (probs >= thr).astype(int)
    auc = roc_auc_score(y, probs)
    prec, rec, f1, _ = precision_recall_fscore_support(y, preds, average='binary', zero_division=0)
    cm = confusion_matrix(y, preds)
    print(f"\033[1m\n{name} metrics @ threshold={thr:.2f}\033[0m")
    print(f"AUC: {auc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    print("\033[1m\nConfusion matrix:\033[0m\n", cm)
    print("\033[1m\nClassification report:\033[0m\n", classification_report(y, preds, digits=4))

# Evaluate
evaluate_split("Validation", X_val, y_val, thr=0.5)
evaluate_split("Test",       X_test, y_test, thr=0.5)


Validation metrics @ threshold=0.50
AUC: 0.9781 | Precision: 0.2025 | Recall: 0.8571 | F1: 0.3276

Confusion matrix:
 [[42476   189]
 [    8    48]]

Classification report:
               precision    recall  f1-score   support

           0     0.9998    0.9956    0.9977     42665
           1     0.2025    0.8571    0.3276        56

    accuracy                         0.9954     42721
   macro avg     0.6012    0.9264    0.6627     42721
weighted avg     0.9988    0.9954    0.9968     42721


Test metrics @ threshold=0.50
AUC: 0.9513 | Precision: 0.2500 | Recall: 0.8077 | F1: 0.3818

Confusion matrix:
 [[42544   126]
 [   10    42]]

Classification report:
               precision    recall  f1-score   support

           0     0.9998    0.9970    0.9984     42670
           1     0.2500    0.8077    0.3818        52

    accuracy                         0.9968     42722
   macro avg     0.6249    0.9024    0.6901     42722
weighted avg     0.9989    0.9968    0.9977     42722

